# 01 - Reproduce the frozen tables and headline numbers (CPU, no GPU)

**Pre-registered pass criteria (written before running).** The notebook PASSES iff all hold:

1. `scripts/verify_tables.py` exits 0 with `MISMATCHES: 0` (every generated table is identical, up to the
   generation date, to the frozen copy, and the regression gate returns 0).
2. ViProKG total MRR recomputed from the per-key ranks (mean over seeds 42/7/2024):
   MENTOR_base+L (lambda=8) = 41.20 and MENTOR_base = 40.89 (absolute difference < 0.005 points).
3. `hits_bang.tsv` carries the signed headline rows: MKG-W 42.50, MKG-Y 44.80, ViProKG 41.20.
   (MKG-W / MKG-Y headline rows are stored constants; per-query ranks for them are not part of this release.)
4. Answer-name masking on ViProKG, recomputed from the shipped per-key dumps, equals the stored result:
   MRR drop 10.78 overall / 17.43 mention-overlap / 8.46 mention-free (equal to the stored JSON to 1e-9).

Any other outcome is FAIL; do not adjust the criteria after seeing the numbers.

In [10]:
import os, glob, subprocess, sys, json, time, hashlib, shutil
INPUT = os.environ.get("KAGGLE_INPUT", "/kaggle/input")

def find_release():
    """Directory of the attached release dataset (contains scripts/verify_tables.py)."""
    hits = sorted(glob.glob(f"{INPUT}/**/scripts/verify_tables.py", recursive=True))
    assert hits, "Attach the release dataset (the folder containing scripts/verify_tables.py)."
    return os.path.dirname(os.path.dirname(hits[0]))

def run_streaming(cmd, cwd=None, env=None, tail=40):
    """Run a command, print its output line by line, return (returncode, last lines)."""
    p = subprocess.Popen(cmd, cwd=cwd, env=env, stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True)
    last = []
    for line in p.stdout:
        print(line, end="", flush=True)
        last.append(line); last = last[-tail:]
    return p.wait(), last

SRC_REL = find_release()
WORK = "/kaggle/working" if os.path.isdir("/kaggle/working") else "/tmp/mentor_nb1"
REL = f"{WORK}/release"
if os.path.exists(REL): shutil.rmtree(REL)
shutil.copytree(SRC_REL, REL)   # input datasets are read-only; the generators write into the release tree
print("release (working copy):", REL)

release (working copy): /kaggle/working/release


## 1. Regenerate all tables and compare with the frozen copies

In [11]:
rc, last = run_streaming([sys.executable, "scripts/verify_tables.py"], cwd=REL)
C1 = (rc == 0 and any("MISMATCHES: 0" in l for l in last))
print("\nCRITERION 1:", "PASS" if C1 else "FAIL")

0 mismatches: every headline cell matches the frozen anchor.
OK   bang_chinh  A(all numbers, 213 tokens)=ok  B(table body, 193 tokens)=ok
OK   bang_vimmkg2  A(all numbers, 145 tokens)=ok  B(table body, 132 tokens)=ok
OK   bang_phuluc_db15k  A(all numbers, 133 tokens)=ok  B(table body, 110 tokens)=ok
OK   bang_trung_tam  A(all numbers, 58 tokens)=ok  B(table body, 33 tokens)=ok
OK   bang_canh_v2  A(all numbers, 50 tokens)=ok  B(table body, 34 tokens)=ok
OK   bang_e_che  A(all numbers, 77 tokens)=ok  B(table body, 40 tokens)=ok
OK   bang_lat_prior  A(all numbers, 49 tokens)=ok  B(table body, 35 tokens)=ok
OK   bang_anh  A(all numbers, 63 tokens)=ok  B(table body, 49 tokens)=ok
OK   bang_e_muc  A(all numbers, 45 tokens)=ok  B(table body, 19 tokens)=ok
OK   bang_prior_k3  A(all numbers, 51 tokens)=ok  B(table body, 39 tokens)=ok
OK   bang_tran_thong_tin  A(all numbers, 41 tokens)=ok  B(table body, 17 tokens)=ok
OK   bang_ablation  A(all numbers, 67 tokens)=ok  B(table body, 45 tokens)=ok
O

## 2. ViProKG total MRR from per-key ranks

In [12]:
import csv, statistics as st
RV = f"{REL}/data/experimental/v3/ranks_v2"
def mrr_wide(p):
    r = []
    for x in csv.DictReader(open(p, encoding="utf-8"), delimiter="\t"):
        r += [1 / float(x["head_rank"]), 1 / float(x["tail_rank"])]
    return st.fmean(r)
def mrr_dump(p):
    return st.fmean(1 / float(x["rank_gold"]) for x in csv.DictReader(open(p, encoding="utf-8"), delimiter="\t"))
SEEDS = (42, 7, 2024)
full = 100 * st.fmean(mrr_wide(f"{RV}/A3pLC_s{s}/ranks_test.tsv") for s in SEEDS)
base = 100 * st.fmean(mrr_dump(f"{RV}/nen_l8_s{s}/dump_test.tsv") for s in SEEDS)
print(f"MENTOR_base+L (lambda=8): {full:.4f}   expected 41.20")
print(f"MENTOR_base            : {base:.4f}   expected 40.89")
C2 = abs(round(full, 2) - 41.20) < 0.005 and abs(round(base, 2) - 40.89) < 0.005
print("CRITERION 2:", "PASS" if C2 else "FAIL")

MENTOR_base+L (lambda=8): 41.2040   expected 41.20
MENTOR_base            : 40.8930   expected 40.89
CRITERION 2: PASS


## 3. Signed headline rows

In [13]:
rows = {}
for l in open(f"{REL}/data/experimental/v3/hits_bang.tsv", encoding="utf-8").read().splitlines()[1:]:
    v = l.split("\t")
    if v[3] == "MRR" and v[2] == "t\u1ed5ng":
        rows[(v[0], v[1])] = float(v[5]) * 100
want = {("MENTOR headline", "MKG-W"): 42.50, ("MENTOR headline", "MKG-Y"): 44.80,
        ("A3\u2032* \u03bb8 (ours)", "ViMMKG2"): 41.20}
C3 = True
for k, w in want.items():
    got = rows.get(k); ok = got is not None and abs(got - w) < 0.005; C3 &= ok
    print(f"{k[1]:8s} {got} expected {w} -> {'ok' if ok else 'MISMATCH'}")
print("CRITERION 3:", "PASS" if C3 else "FAIL")

MKG-W    42.5 expected 42.5 -> ok
MKG-Y    44.800000000000004 expected 44.8 -> ok
ViMMKG2  41.204 expected 41.2 -> ok
CRITERION 3: PASS


## 4. Answer-name masking (ViProKG)

In [14]:
rc, _ = run_streaming([sys.executable, "scripts/score_masking_drop.py", "--out_json", "/kaggle/working/mask_drop.json"
                       if os.path.isdir("/kaggle/working") else "/tmp/mask_drop.json"], cwd=REL)
got = json.load(open("/kaggle/working/mask_drop.json" if os.path.isdir("/kaggle/working") else "/tmp/mask_drop.json"))
ref = json.load(open(f"{REL}/data/experimental/v3/che_v2_dumps/che2_kq.json"))
pairs = [("drop_total", "rot_tong"), ("drop_overlap", "rot_co"), ("drop_free", "rot_khong")]
C4 = rc == 0 and all(abs(got[a] - ref[b]) < 1e-9 for a, b in pairs)
print({a: round(got[a], 3) for a, _ in pairs}, "ratio", round(got["ratio"], 2))
print("CRITERION 4:", "PASS" if C4 else "FAIL")

{
 "drop_total": 10.782804679770841,
 "drop_overlap": 17.42920756279373,
 "drop_free": 8.464212983493798,
 "ratio": 2.0591645787721453
}
{'drop_total': 10.783, 'drop_overlap': 17.429, 'drop_free': 8.464} ratio 2.06
CRITERION 4: PASS


In [15]:
ALL = C1 and C2 and C3 and C4
print("OVERALL:", "PASS" if ALL else "FAIL", {"C1": C1, "C2": C2, "C3": C3, "C4": C4})

OVERALL: PASS {'C1': True, 'C2': True, 'C3': True, 'C4': True}
